created with the help of copilot

## RQ8: How does a high density off gas stations impact the fuel prices?

In [1]:
import polars as pl
import plotly.express as px
import numpy as np
from sklearn.cluster import DBSCAN
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
import glob

import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.RQ_8_competition_script import save_png
from scripts.RQ_8_competition_script import getCSVData
from scripts.RQ_8_competition_script import plotClusters
from scripts.RQ_8_competition_script import performDBSCAN
from scripts.RQ_8_competition_script import create_CSV_with_cluster_labels
from scripts.RQ_8_competition_script import join_labels_and_group 
from scripts.RQ_8_competition_script import plot_cluster_prices
from scripts.RQ_8_competition_script import analyse_motorway_clusters
from scripts.RQ_8_competition_script import plot_cluster_difference
from scripts.RQ_8_competition_script import compute_cluster_counts_over_time
from scripts.RQ_8_competition_script import plot_motorway_cluster_pies
from scripts.RQ_8_competition_script import plot_cluster_counts_over_time
from scripts.RQ_8_competition_script import plot_yearly_boxplot
from scripts.rq8_build_station_daily_by_month import build_station_daily_by_month

In [2]:
INPUT_ROOT = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\PersonalTesting\\dbscan"
OUTPUT_ROOT = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\PersonalTesting\\dbscan\\labeled_stations"
INPUT_PRICES = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\TankKoenigData"
OUTPUT_PRICES = "C:\\Users\\Bjarne\\Desktop\\Uni\\Data Science Projekt\\PersonalTesting\\dbscan"

In [3]:
#read the data from the CSV file
df = getCSVData(INPUT_ROOT + "\\stations.csv")

shape: (15_442, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ uuid      ┆ name      ┆ brand     ┆ street    ┆ … ┆ post_code ┆ city      ┆ latitude  ┆ longitud │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ e        │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ i64       ┆ str       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 00060723- ┆ BAGeno    ┆ null      ┆ Künzelsau ┆ … ┆ 74653     ┆ Ingelfing ┆ 49.296822 ┆ 9.661385 │
│ 0001-4444 ┆ Raiffeise ┆           ┆ er        ┆   ┆           ┆ en        ┆           ┆          │
│ -8888-acd ┆ n eG      ┆           ┆ Strasse   ┆   ┆           ┆           ┆           ┆          │
│ c00…      ┆           ┆           ┆           ┆   ┆           ┆       

In [4]:
#parameter tuning for DBSCAN

eps1 = 2 #in km
min_samples1 = 4

eps2 = 0.2 #in km
min_samples2 = 2

# execute DBSCAN and plot results
labels1 = performDBSCAN(df, eps1, min_samples1)
figure1 = plotClusters(df, labels1)
figure1.show()

labels2 = performDBSCAN(df, eps2, min_samples2)
figure2 = plotClusters(df, labels2)
save_png(figure2, "dbscan_clusters_eps.png", True)

figure2.show()

Number of clusters: 799
[798  -1   0 ...  -1  -1 536]


Number of clusters: 1138
[  -1   -1   -1 ...  449 1020 1100]


In [5]:
clusters = pl.read_csv(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv"
)

noise_count_esp1 = clusters.filter(pl.col("cluster") == -1).height

clusters = pl.read_csv(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv"
)

noise_count_esp2 = clusters.filter(pl.col("cluster") == -1).height

print("Noise stations Cluster 1:", noise_count_esp1)
print("Noise stations Cluster 2:", noise_count_esp2)

Noise stations Cluster 1: 7631
Noise stations Cluster 2: 13038


In [6]:
create_CSV_with_cluster_labels(
    df,    
    labels1, 
    OUTPUT_ROOT + "\\stations_clusters" + f"_eps{eps1}_min_samples{min_samples1}.csv"
    )

create_CSV_with_cluster_labels(
    df, 
    labels2, 
    OUTPUT_ROOT + "\\stations_clusters" + f"_eps{eps2}_min_samples{min_samples2}.csv"
    )

In [7]:
df_clusters1 = pl.read_csv(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv"
)
df_clusters2 = pl.read_csv(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv"
)

plotClusters(df_clusters1, df_clusters1["cluster"]).show()
plotClusters(df_clusters2, df_clusters2["cluster"]).show()

In [8]:
# Call the build_station_daily_by_month function with proper parameters
# INPUT_ROOT should contain prices/{year}/{month}/ subdirectories
build_station_daily_by_month(
    data_root=INPUT_PRICES,
    derived_root=OUTPUT_PRICES,
    start_year=2014,
    end_year=2026
)

[2014-01] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\01\*-prices.csv
[2014-01] no files / unreadable -> skip
[2014-02] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\02\*-prices.csv
[2014-02] no files / unreadable -> skip
[2014-03] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\03\*-prices.csv
[2014-03] no files / unreadable -> skip
[2014-04] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\04\*-prices.csv
[2014-04] no files / unreadable -> skip
[2014-05] reading: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\TankKoenigData\prices\2014\05\*-prices.csv
[2014-05] no files / unreadable -> skip
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2014\2014-06.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_dai

Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2016\2016-09.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2016\2016-10.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2016\2016-11.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2016\2016-12.parquet
[2016] DONE
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2017\2017-01.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2017\2017-02.parquet
Skip (exists): C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_

In [9]:
# Displays a sample of the resulting parquet files to verify the structure and content.

parquet_dir = Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month"
parquet_files = sorted(glob.glob(str(parquet_dir / "**" / "*.parquet"), recursive=True))

if parquet_files:
    df_sample = pl.read_parquet(parquet_files[0])
    print(f"Table structure from: {parquet_files[0]}\n")
    print(df_sample.head(10))
    print(f"\nColumns: {df_sample.columns}")
    print(f"Shape: {df_sample.shape}")
else:
    print("No parquet files found. Run the previous cell first.")

Table structure from: C:\Users\Bjarne\Desktop\Uni\Data Science Projekt\PersonalTesting\dbscan\station_daily_mean_and_median_by_month\2014\2014-06.parquet

shape: (10, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ station_uu ┆ day       ┆ diesel_me ┆ diesel_me ┆ … ┆ e5_median ┆ e10_mean ┆ e10_media ┆ n_events │
│ id         ┆ ---       ┆ an        ┆ dian      ┆   ┆ ---       ┆ ---      ┆ n         ┆ ---      │
│ ---        ┆ date      ┆ ---       ┆ ---       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ u32      │
│ str        ┆           ┆ f64       ┆ f64       ┆   ┆           ┆          ┆ f64       ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ 00041450-0 ┆ 2014-06-0 ┆ 1.318     ┆ 1.318     ┆ … ┆ 1.538     ┆ 1.498    ┆ 1.498     ┆ 1        │
│ 002-4444-8 ┆ 8         ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ 888-acdc00 ┆        

In [10]:
daily_prices = pl.concat([pl.read_parquet(f) for f in parquet_files])
print(daily_prices.head())

shape: (5, 9)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬──────────┐
│ station_uu ┆ day       ┆ diesel_me ┆ diesel_me ┆ … ┆ e5_median ┆ e10_mean ┆ e10_media ┆ n_events │
│ id         ┆ ---       ┆ an        ┆ dian      ┆   ┆ ---       ┆ ---      ┆ n         ┆ ---      │
│ ---        ┆ date      ┆ ---       ┆ ---       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ u32      │
│ str        ┆           ┆ f64       ┆ f64       ┆   ┆           ┆          ┆ f64       ┆          │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪══════════╡
│ 00041450-0 ┆ 2014-06-0 ┆ 1.318     ┆ 1.318     ┆ … ┆ 1.538     ┆ 1.498    ┆ 1.498     ┆ 1        │
│ 002-4444-8 ┆ 8         ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ 888-acdc00 ┆           ┆           ┆           ┆   ┆           ┆          ┆           ┆          │
│ …          ┆           ┆           ┆           ┆   ┆           ┆          ┆

Now we will start plotting and analyzing the Data

In [11]:
# Choose the Fuel Type to analyze, from: diesel, e5, e10
FUEL_TYPE = "e5"

In [12]:
parquet_dir = Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month"
parquet_files = sorted(glob.glob(str(parquet_dir / "**" / "*.parquet"), recursive=True))


# Diesel, Cluster Set 1
fig_1 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    fuel = FUEL_TYPE,
    title = FUEL_TYPE + "Prices - Cluster Set 1"
)
fig_1.show()

fig_2 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    fuel = FUEL_TYPE,
    title = FUEL_TYPE + "Prices - Cluster Set 2"
)
fig_2.show()

Now we will start cleaning all the Motorwaystations from the Dataset, to prevent scewed results.

In [13]:
motorway_stations = (
    pl.read_csv(
        INPUT_ROOT + "\\autobahn_stations.csv",
        null_values=["nicht", "NA", "N/A", "", "Nicht"]
    )
    .select(pl.col("uuid").alias("station_uuid"))
)

print(motorway_stations.head())
print("Motorway stations:", motorway_stations.height)

shape: (5, 1)
┌─────────────────────────────────┐
│ station_uuid                    │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ 005056ba-7cb6-1ed2-bceb-662ba1… │
│ 29cd2450-3240-4728-a066-ded2af… │
│ 3bfab341-1f1e-49e0-8c32-bc77ca… │
│ 3f1057c6-aeb9-44d8-9914-645c8b… │
│ 4947beb1-79b7-4f23-85f9-24fcd7… │
└─────────────────────────────────┘
Motorway stations: 376


In [14]:
# Plot the distribution of motorway stations across clusters for both parameter sets.
fig = plot_motorway_cluster_pies(
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    motorway_stations
)

fig.show()

Now we will plot the cleaned data from both cluster sets.

In [ ]:
fig_diesel_clean_esp1 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title = FUEL_TYPE + "Prices - Cluster Set 1 (no motorway stations)"
) 

fig_diesel_clean_esp2 = plot_cluster_prices(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title = FUEL_TYPE + "Prices - Cluster Set 2 (no motorway stations)"
)   



fig_diesel_clean_esp1.show()
fig_diesel_clean_esp2.show()

For easier visual understanding we will now only plot the difference between the clustered und unclustered data. 
Positive numbers indicate a a higher price from the clustered data, for example: 0.30 ==> The clustered Stations costs 30 cents more. Negative Numbers on the other hand indicate that clustered stations are cheaper, than the unclustered.

In [ ]:
parquet_dir = Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month"
parquet_files = sorted(glob.glob(str(parquet_dir / "**" / "*.parquet"), recursive=True))

fig_diff, diff_df1 = plot_cluster_difference(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title=FUEL_TYPE + " Cluster 1 vs Noise Difference (Mean & Median, no motorway)"
)
fig_diff.show()

diff_df1.to_csv(OUTPUT_ROOT + f"\\cluster_1_diff_{FUEL_TYPE}.csv", index=False)

fig_diff, diff_df2 = plot_cluster_difference(
    parquet_files,
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv",
    fuel = FUEL_TYPE,
    motorway_df=motorway_stations,
    title = FUEL_TYPE + " Cluster 2 vs Noise Difference (Mean & Median, no motorway)"
)
fig_diff.show()

diff_df2.to_csv(OUTPUT_ROOT + f"\\cluster_2_diff_{FUEL_TYPE}.csv", index=False)

In [ ]:
fig_cluster1 = plot_yearly_boxplot(diff_df1, "Cluster Set 1", FUEL_TYPE)
fig_cluster1.show()

fig_cluster2 = plot_yearly_boxplot(diff_df2, "Cluster Set 2", FUEL_TYPE)
save_png(fig_cluster2, OUTPUT_ROOT + f"\\cluster_difference_yearly_boxplot_eps{eps2}_min_samples{min_samples2}.png")
fig_cluster2.show()

In [ ]:
counts_eps1 = compute_cluster_counts_over_time(
    daily_prices, 
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv"
)
counts_eps2 = compute_cluster_counts_over_time(
    daily_prices, 
    OUTPUT_ROOT + f"\\stations_clusters_eps{eps2}_min_samples{min_samples2}.csv"
)


In [ ]:
fig_counts_eps1 = plot_cluster_counts_over_time(counts_eps1, title="Cluster Set 1: Cluster vs Noise")
fig_counts_eps1.show()

fig_counts_eps2 = plot_cluster_counts_over_time(counts_eps2, title="Cluster Set 2: Cluster vs Noise")
fig_counts_eps2.show()

Evtl obsolet:


In [ ]:
# Load cluster labels
clusters = (
    pl.read_csv(
        OUTPUT_ROOT + f"\\stations_clusters_eps{eps1}_min_samples{min_samples1}.csv"
    )
    .rename({"uuid": "station_uuid"})
    .select(["station_uuid", "cluster"])
    .lazy()
)

In [ ]:
# Load all daily station price data

daily_prices = pl.scan_parquet(
    str(Path(OUTPUT_PRICES) / "station_daily_mean_and_median_by_month" / "**/*.parquet")
)

print(daily_prices)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Parquet SCAN [C:/Users/Bjarne/Desktop/Uni/Data Science Projekt/PersonalTesting/dbscan/station_daily_mean_and_median_by_month/2014/2014-06.parquet, ... 140 other sources]
PROJECT */9 COLUMNS
ESTIMATED ROWS: 44159649


In [ ]:
df = daily_prices.join(
    clusters,
    on="station_uuid",
    how="left"
)

In [ ]:
df = df.with_columns(
    pl.when(pl.col("cluster") == -1)
    .then(pl.lit("noise"))
    .otherwise(pl.lit("cluster"))
    .alias("group")
)

In [ ]:
start_year = 2014
end_year = 2024

df_years = df.filter(
    (pl.col("day").dt.year() >= start_year) &
    (pl.col("day").dt.year() <= end_year)
)

In [ ]:
FUEL_TYPE = "e5"
daily_groups = (
    df_years
    .group_by(["day", "group"])
    .agg([
        pl.col(FUEL_TYPE + "_mean").mean().alias(FUEL_TYPE + "_mean_price"),
        pl.col(FUEL_TYPE + "_median").mean().alias(FUEL_TYPE + "_median_price")
    ])
    .sort("day")
    .collect()
)